{ “cells”: \[ { “cell_type”: “code”, “execution_count”: null,
“metadata”: {}, “outputs”: \[\], “source”: \[ “{”, ” "cells": \[“,” {“,”
"cell_type": "markdown",“,” "metadata": {},“,” "source": \[“,” "# 🏃‍♂️
Endurance Sports RAG: Pure Python Agent\n",“,” "\n",“,” "**The Blunt
Truth:** A vector database is a dumb filing cabinet. If you pass
conversational text into it, it will fail. \n",“,” "This notebook
implements a modern RAG pipeline from scratch:\n",“,” "1. **Query
Rewriting:** An LLM translates your chat message into clinical search
terms.\n",“,” "2. **Two-Stage Retrieval:** OpenAI Vectors (Stage 1) +
HuggingFace Reranker (Stage 2).\n",“,” "3. **Synthesis:** An LLM reads
the top chunks and acts as your coach.\n",“,” "4. **Gradio UI:** A clean
interface to chat with the system."“,” \]“,” },“,” {“,” "cell_type":
"code",“,” "execution_count": null,“,” "metadata": {},“,” "outputs":
\[\],“,” "source": \[“,” "# Cell 1: Install Dependencies\n",“,” "!pip
install openai chromadb sentence-transformers gradio"“,” \]“,” },“,”
{“,” "cell_type": "code",“,” "execution_count": null,“,” "metadata":
{},“,” "outputs": \[\],“,” "source": \[“,” "# Cell 2: Setup &
Authentication\n",“,” "import os\n",“,” "import openai\n",“,” "import
chromadb\n",“,” "from chromadb.utils import embedding_functions\n",“,”
"from sentence_transformers import CrossEncoder\n",“,” "import gradio as
gr\n",“,” "\n",“,” "# Ensure your API keys are set in your environment
variables\n",“,” "OPENAI_KEY =
os.environ.get(\\"OPENAI_API_KEY\\")\n",“,” "if not OPENAI_KEY:\n",“,” "
raise ValueError(\\"❌ OPENAI_API_KEY is missing. Set it before
continuing.\\")\n",“,” "\n",“,” "# Initialize OpenAI Client\n",“,”
"client = openai.OpenAI(api_key=OPENAI_KEY)\n",“,” "print(\\"✅ APIs
configured successfully.\\")"“,” \]“,” },“,” {“,” "cell_type":
"code",“,” "execution_count": null,“,” "metadata": {},“,” "outputs":
\[\],“,” "source": \[“,” "# Cell 3: Connect to the existing Vector
Database\n",“,” "DB_PATH = \\"./endurance_db\\"\n",“,” "COLLECTION_NAME
= \\"endurance_knowledge_base\\"\n",“,” "\n",“,” "print(\\"🧠 Connecting
to Local Vector Database…\\")\n",“,” "chroma_client =
chromadb.PersistentClient(path=DB_PATH)\n",“,” "\n",“,” "openai_ef =
embedding_functions.OpenAIEmbeddingFunction(\n",“,” "
api_key=OPENAI_KEY,\n",“,” "
model_name=\\"text-embedding-3-small\\"\n",“,” ")\n",“,” "\n",“,”
"collection = chroma_client.get_collection(\n",“,” "
name=COLLECTION_NAME,\n",“,” " embedding_function=openai_ef\n",“,”
")\n",“,” "\n",“,” "# Load Reranker\n",“,” "print(\\"⚖️ Loading Hugging
Face Reranker…\\")\n",“,” "reranker =
CrossEncoder(‘BAAI/bge-reranker-base’)\n",“,” "\n",“,” "print(f\\"✅
Connected! Database currently holds {collection.count()} chunks.\\")"“,”
\]“,” },“,” {“,” "cell_type": "code",“,” "execution_count": null,“,”
"metadata": {},“,” "outputs": \[\],“,” "source": \[“,” "# Cell 4: The
Query Rewriter (The Brain Upgrade)\n",“,” "# This function fixes the
‘Semantic Gap’ by converting chatty messages into clinical
searches.\n",“,” "\n",“,” "def rewrite_query(user_message):\n",“,” "
prompt = f\\"\\"\\"\n",“,” " You are an expert endurance sports data
engineer.\n",“,” " Your job is to convert the user’s conversational
message into a highly specific, clinical search query \n",“,” "
optimized for a vector database containing physiological metrics (HR,
Pace, Distance) and scientific sports abstracts.\n",“,” " \n",“,” " User
message: ‘{user_message}’\n",“,” " \n",“,” " Output ONLY the rewritten
search query. Do not include any conversational filler.\n",“,” "
\\"\\"\\"\n",“,” " \n",“,” " response =
client.chat.completions.create(\n",“,” " model=\\"gpt-4o-mini\\", \#
Fast and cheap for this intermediate step\n",“,” "
messages=\[{\\"role\\": \\"user\\", \\"content\\": prompt}\],\n",“,” "
temperature=0.1\n",“,” " )\n",“,” " \n",“,” " rewritten =
response.choices\[0\].message.content.strip()\n",“,” " return
rewritten\n",“,” "\n",“,” "# — Let’s test it immediately —\n",“,”
"test_msg = \\"I had a really tiring, long distance run
recently.\\"\n",“,” "print(f\\"Original: {test_msg}\\")\n",“,”
"print(f\\"Rewritten: {rewrite_query(test_msg)}\\")"“,” \]“,” },“,” {“,”
"cell_type": "code",“,” "execution_count": null,“,” "metadata": {},“,”
"outputs": \[\],“,” "source": \[“,” "# Cell 5: Two-Stage Retrieval
Function\n",“,” "def retrieve_and_rerank(search_query, top_k=3):\n",“,”
" \# Stage 1: Fast Coarse Search\n",“,” " results =
collection.query(\n",“,” " query_texts=\[search_query\],\n",“,” "
n_results=15\n",“,” " )\n",“,” " candidates =
results\[‘documents’\]\[0\]\n",“,” " metadatas =
results\[‘metadatas’\]\[0\]\n",“,” " \n",“,” " if not candidates:\n",“,”
" return \\"\\"\n",“,” " \n",“,” " \# Stage 2: Cross-Encoder
Reranking\n",“,” " pairs = \[\[search_query, doc\] for doc in
candidates\]\n",“,” " scores = reranker.predict(pairs)\n",“,” " \n",“,”
" \# Sort by reranker score\n",“,” " ranked_results = sorted(zip(scores,
candidates, metadatas), key=lambda x: x\[0\], reverse=True)\n",“,” "
\n",“,” " \# Extract the absolute best chunks\n",“,” " context_blocks =
\[\]\n",“,” " for score, doc, meta in ranked_results\[:top_k\]:\n",“,” "
context_blocks.append(f\\"\[Source: {meta.get(‘source’, ‘Unknown’)} \|
Type: {meta.get(‘document_type’, ‘Unknown’)}\]\\\n{doc}\\")\n",“,” "
\n",“,” " return \\"\\\n\\\n—\\\n\\\n\\".join(context_blocks)"“,” \]“,”
},“,” {“,” "cell_type": "code",“,” "execution_count": null,“,”
"metadata": {},“,” "outputs": \[\],“,” "source": \[“,” "# Cell 6: The
Final Generation Step\n",“,” "def generate_response(user_message,
chat_history):\n",“,” " \# 1. Rewrite the query\n",“,” " search_query =
rewrite_query(user_message)\n",“,” " print(f\\"\\\n\[DEBUG\] Rewritten
Query: {search_query}\\")\n",“,” " \n",“,” " \# 2. Retrieve
Context\n",“,” " context = retrieve_and_rerank(search_query)\n",“,” "
print(f\\"\[DEBUG\] Retrieved Context:\\\n{context}\\")\n",“,” " \n",“,”
" \# 3. Build the System Prompt\n",“,” " system_prompt =
f\\"\\"\\"\n",“,” " You are an elite endurance sports coach and data
analyst. \n",“,” " You must answer the user’s question using ONLY the
physiological data and scientific studies provided in the context
below.\n",“,” " If the context does not contain the answer, bluntly
state that you do not have the data to answer it. \n",“,” " Do not
hallucinate fitness advice.\n",“,” " \n",“,” " CONTEXT:\n",“,” "
{context}\n",“,” " \\"\\"\\"\n",“,” " \n",“,” " \# Format history for
OpenAI\n",“,” " messages = \[{\\"role\\": \\"system\\", \\"content\\":
system_prompt}\]\n",“,” " for past_user, past_bot in
chat_history:\n",“,” " messages.append({\\"role\\": \\"user\\",
\\"content\\": past_user})\n",“,” " messages.append({\\"role\\":
\\"assistant\\", \\"content\\": past_bot})\n",“,” " \n",“,” "
messages.append({\\"role\\": \\"user\\", \\"content\\":
user_message})\n",“,” " \n",“,” " \# 4. Generate Answer\n",“,” "
response = client.chat.completions.create(\n",“,” " model=\\"gpt-4o\\",
\# Use the smart model for final synthesis\n",“,” "
messages=messages,\n",“,” " temperature=0.3\n",“,” " )\n",“,” " \n",“,”
" return response.choices\[0\].message.content"“,” \]“,” },“,” {“,”
"cell_type": "code",“,” "execution_count": null,“,” "metadata": {},“,”
"outputs": \[\],“,” "source": \[“,” "# Cell 7: Launch the Gradio
UI\n",“,” "# Note: Running this cell will block the notebook and open a
chat UI in the output.\n",“,” "\n",“,” "print(\\"🚀 Launching
UI…\\")\n",“,” "demo = gr.ChatInterface(\n",“,” "
fn=generate_response,\n",“,” " title=\\"🏃‍♂️ AI Endurance Coach (Pure
Python RAG)\\",\n",“,” " description=\\"Ask about your physiological
metrics or endurance science. I use a 2-stage retrieval pipeline to
fetch data before answering.\\",\n",“,” " examples=\[\n",“,” " \\"I had
a really tiring, long distance run recently. What does the data
say?\\",\n",“,” " \\"What scientific literature do you have on
tapering?\\",\n",“,” " \\"Based on my logs, what is my average heart
rate on long runs?\\"\n",“,” " \]\n",“,” ")\n",“,” "\n",“,”
"demo.launch(share=False) \# Set share=True if you want a public
link"“,” \]“,” }“,” \],“,” "metadata": {“,” "kernelspec": {“,”
"display_name": "Python 3",“,” "language": "python",“,” "name":
"python3"“,” },“,” "language_info": {“,” "codemirror_mode": {“,” "name":
"ipython",“,” "version": 3“,” },“,” "file_extension": ".py",“,”
"mimetype": "text/x-python",“,” "name": "python",“,”
"nbconvert_exporter": "python",“,” "pygments_lexer": "ipython3",“,”
"version": "3.9.0"“,” }“,” },“,” "nbformat": 4,“,” "nbformat_minor":
4“,”}” \] }, { “cell_type”: “code”, “execution_count”: null, “metadata”:
{}, “outputs”: \[\], “source”: \[\] } \], “metadata”: { “language_info”:
{ “name”: “python” } }, “nbformat”: 4, “nbformat_minor”: 2 }